# Day 4 - Pandas Full Workflow Colab Examples

This notebook is a Google Colab friendly worked example of a complete pandas workflow:
`load -> inspect -> clean -> filter -> combine -> summarize -> reshape -> visualize -> export`.

It keeps the same Day 4 theory and the same demo datasets as the local working notebook, but it is now hybrid:
- if local `data/day4/raw/` files exist, it uses them directly;
- otherwise it loads the source files from this GitHub repository where appropriate:
`https://github.com/ValRCS/RTU_Python_CSP`.

Notebook behavior:
- The setup cell detects whether local Day 4 data is available.
- In local mode, inputs are read from the repository and outputs are written into `data/day4/outputs/`.
- In remote mode, CSV, JSON, and HTML inputs are loaded from GitHub raw URLs.
- In remote mode, binary files such as `SQLite` and `Excel` are first downloaded into a runtime cache, then read locally.
- The final cells generate exports in the active output folder.

Important note:
- The raw URLs target the `main` branch of this repository.
- If you update the local `data/day4/raw/` files, push them to GitHub before relying on this Colab notebook.

Recommended usage:
1. Open the notebook in Google Colab.
2. Or run it locally in Jupyter from the repository.
3. Run the notebook from top to bottom.
4. Compare the code to the theory notes and adjust parameters interactively if needed.
5. Inspect the generated files in the active output folder.


## Workflow Map

A strong pandas workflow is iterative rather than perfectly linear, but this order is usually the most reliable:
- `Load`: bring source data into one or more DataFrames with as little accidental distortion as possible.
- `Inspect`: learn the table grain, schema, data types, missingness, duplicates, and likely problem areas.
- `Clean`: standardize names, types, values, categories, and row-level quality issues.
- `Filter`: keep only the rows and columns required for the current question.
- `Combine`: merge related tables or stack repeated extracts.
- `Summarize`: produce grouped metrics, pivot-style reports, and decision-ready aggregates.
- `Reshape`: convert between wide and long layouts depending on reporting and plotting needs.
- `Visualize`: turn prepared tables into charts that answer a specific question.
- `Export`: save cleaned data, summary tables, and outputs in reusable formats.

Useful mental model:
- Early stages protect data fidelity.
- Middle stages create analysis-ready tables.
- Late stages communicate and deliver results.

Also expect to loop back:
- Inspection may show that the data should be loaded with different parameters.
- Cleaning may reveal that filters need to be delayed or revised.
- Combining may expose key mismatches that require more cleaning.
- Visualization often reveals that a summary or reshape step should change.


In [ ]:
from __future__ import annotations

import importlib.util
import sqlite3
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
    "lxml": "lxml",
    "bs4": "beautifulsoup4",
    "html5lib": "html5lib",
}
missing_packages = sorted(
    {package for module_name, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None}
)
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

REPO_WEB_URL = "https://github.com/ValRCS/RTU_Python_CSP"
RAW_BASE_URL = "https://raw.githubusercontent.com/ValRCS/RTU_Python_CSP/main/data/day4/raw"

def find_repo_root() -> Path | None:
    candidates: list[Path] = []

    try:
        candidates.append(Path(__file__).resolve().parents[1])
    except NameError:
        pass

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd.parent, cwd.parent.parent if len(cwd.parents) >= 2 else cwd])

    for candidate in candidates:
        if (candidate / "data" / "day4" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate

    return None


LOCAL_REPO_ROOT = find_repo_root()
USE_LOCAL_DATA = LOCAL_REPO_ROOT is not None
IN_COLAB = "google.colab" in sys.modules

if USE_LOCAL_DATA:
    DATA_DIR = LOCAL_REPO_ROOT / "data" / "day4"
    RAW_DIR = DATA_DIR / "raw"
    INTERIM_DIR = DATA_DIR / "interim"
    OUTPUT_DIR = DATA_DIR / "outputs"
    RAW_CACHE_DIR = RAW_DIR
    DATA_SOURCE_MODE = "local"
else:
    RUNTIME_ROOT = Path.cwd() / "day4_colab_runtime"
    RAW_CACHE_DIR = RUNTIME_ROOT / "raw_cache"
    INTERIM_DIR = RUNTIME_ROOT / "interim"
    OUTPUT_DIR = RUNTIME_ROOT / "outputs"
    DATA_SOURCE_MODE = "remote"

for path in (RAW_CACHE_DIR, INTERIM_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 2)

EXCEL_ENGINE = "openpyxl"

if USE_LOCAL_DATA:
    demo_files = {
        "sales_january_csv": RAW_DIR / "sales_january_raw.csv",
        "sales_february_csv": RAW_DIR / "sales_february_raw.csv",
        "products_json": RAW_DIR / "products_catalog.json",
        "stores_html": RAW_DIR / "stores.html",
        "stores_csv": RAW_DIR / "stores.csv",
        "targets_sqlite": RAW_DIR / "regional_targets.sqlite",
        "reference_excel": RAW_DIR / "reference_tables.xlsx",
    }
else:
    demo_files = {
        "sales_january_csv": f"{RAW_BASE_URL}/sales_january_raw.csv",
        "sales_february_csv": f"{RAW_BASE_URL}/sales_february_raw.csv",
        "products_json": f"{RAW_BASE_URL}/products_catalog.json",
        "stores_html": f"{RAW_BASE_URL}/stores.html",
        "stores_csv": f"{RAW_BASE_URL}/stores.csv",
        "targets_sqlite": f"{RAW_BASE_URL}/regional_targets.sqlite",
        "reference_excel": f"{RAW_BASE_URL}/reference_tables.xlsx",
    }

{
    "data_source_mode": DATA_SOURCE_MODE,
    "in_colab": IN_COLAB,
    "output_dir": OUTPUT_DIR,
    "demo_files": demo_files,
}


## 1. Load

Goal: bring source data into pandas with minimal accidental transformation.

Loading is not just an import step. It is where you define how pandas should interpret separators, headers, data types, missing values, dates, decimal symbols, indexes, sheet names, query results, and file encodings. Good loading choices reduce downstream cleaning work and preserve important information such as leading zeros, categorical labels, and date precision.

Common source patterns to demonstrate:
- Delimited text with `pd.read_csv()`, `pd.read_table()`, or `pd.read_fwf()`.
- Spreadsheet files with `pd.read_excel()` from one sheet, many sheets, or a sheet dictionary.
- Semi-structured text with `pd.read_json()`, `pd.json_normalize()`, `pd.read_html()`, or `pd.read_xml()`.
- Columnar and binary formats with `pd.read_parquet()`, `pd.read_feather()`, or `pd.read_pickle()`.
- Databases with `pd.read_sql_query()`, `pd.read_sql_table()`, or `pd.read_sql()`.
- Quick manual sources with `pd.read_clipboard()` or a DataFrame created from Python lists, dictionaries, or API responses.

Parameters worth discussing early:
- `usecols`: load only the columns you actually need.
- `dtype`: prevent pandas from guessing incorrectly, especially for IDs and codes.
- `parse_dates`: parse dates at load time when the format is dependable.
- `index_col`: choose a meaningful index only when it helps later steps.
- `nrows` or `chunksize`: useful for previews and large files.
- `na_values` and `keep_default_na`: define what counts as missing.
- `encoding`, `sep`, `decimal`, and `thousands`: essential for locale-sensitive files.
- `sheet_name`: useful for multi-sheet Excel workbooks.
- storage or engine options: especially relevant for Excel, parquet, and remote storage.

Questions to answer while loading:
- What does one row represent?
- Which columns are mandatory for the analysis question?
- Are there columns that must stay as text even if they look numeric?
- Are there multiple source files, monthly extracts, or sheets that should later be combined?
- Is the data already flat, or will nested structures need normalization?
- Should the first pass load all rows, a small preview, or chunks?

Common pitfalls:
- Losing leading zeros in IDs by allowing automatic numeric conversion.
- Treating locale-specific numbers like text because decimal or thousands symbols were ignored.
- Pulling far more columns than needed, which increases memory use and confusion.
- Assuming JSON is already tabular when nested objects or lists still need flattening.
- Forgetting that SQL can be part of the pandas workflow, not a separate universe.

Documentation references:
- [pandas IO tools user guide](https://pandas.pydata.org/docs/user_guide/io.html)
- [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [pandas.read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html)
- [pandas.read_sql_query](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)
- [pandas.read_parquet](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)


In [ ]:
def ensure_local_binary(source: str | Path, filename: str) -> Path:
    if isinstance(source, Path):
        return source
    destination = RAW_CACHE_DIR / filename
    urlretrieve(source, destination)
    return destination


january_csv_source = demo_files["sales_january_csv"]
february_csv_source = demo_files["sales_february_csv"]
products_json_source = demo_files["products_json"]
stores_html_source = demo_files["stores_html"]
stores_csv_source = demo_files["stores_csv"]
sqlite_source = demo_files["targets_sqlite"]
excel_source = demo_files["reference_excel"]

sales_january_raw = pd.read_csv(january_csv_source)
sales_february_raw = pd.read_csv(february_csv_source)
products_df = pd.read_json(products_json_source)

try:
    stores_df = pd.read_html(stores_html_source)[0]
    stores_source = "HTML table"
except (ImportError, ValueError):
    stores_df = pd.read_csv(stores_csv_source)
    stores_source = "CSV fallback"

sqlite_cache_path = ensure_local_binary(sqlite_source, "regional_targets.sqlite")
with sqlite3.connect(sqlite_cache_path) as connection:
    targets_df = pd.read_sql_query(
        "SELECT month, region, target_revenue FROM regional_monthly_targets ORDER BY month, region",
        connection,
    )

excel_cache_path = ensure_local_binary(excel_source, "reference_tables.xlsx")
excel_tables = pd.read_excel(excel_cache_path, sheet_name=None)

loaded_objects = pd.DataFrame(
    [
        {"dataset": "sales_january_raw", "rows": len(sales_january_raw), "columns": sales_january_raw.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": "sales_february_raw", "rows": len(sales_february_raw), "columns": sales_february_raw.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": "products_df", "rows": len(products_df), "columns": products_df.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": f"stores_df ({stores_source})", "rows": len(stores_df), "columns": stores_df.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": "targets_df", "rows": len(targets_df), "columns": targets_df.shape[1], "source_mode": DATA_SOURCE_MODE},
        {"dataset": "excel_tables", "rows": len(excel_tables), "columns": len(excel_tables), "source_mode": DATA_SOURCE_MODE},
    ]
)
loaded_objects


## 2. Inspect

Goal: understand what was loaded before changing anything.

Inspection is the stage where you learn the shape, grain, and reliability of the data. It should happen immediately after loading and before heavy cleaning. If you skip inspection, you can easily spend time cleaning the wrong columns, filtering on the wrong assumptions, or merging on unstable keys.

Fast first-pass checks:
- Preview rows with `.head()`, `.tail()`, and `.sample()`.
- Check dimensions with `.shape`.
- Review schema with `.columns`, `.dtypes`, and `.info()`.
- Generate numeric and categorical summaries with `.describe()` and `.value_counts()`.
- Measure missingness with `.isna().sum()`.
- Check uniqueness and possible keys with `.nunique()` or duplicate tests.
- Review memory footprint with `.memory_usage(deep=True)` when size matters.

Questions inspection should answer:
- What is the grain of the table: transaction, person, product, date, event, or something else?
- Which columns look like identifiers or future join keys?
- Which columns are unexpectedly `object` or `string` and may need conversion?
- Which fields contain obvious missing values, placeholders, or inconsistent categories?
- Are there duplicate rows or duplicate keys?
- Do any numeric columns show impossible values, suspicious zeros, or very large outliers?
- Which fields are good candidates for grouping, filtering, or plotting later?

Useful teaching angle:
- Treat inspection as a form of hypothesis building.
- Write down what you think each important column means.
- Mark anything uncertain so later cleaning rules are explicit rather than accidental.

Common pitfalls:
- Relying only on `.head()` and assuming the whole dataset behaves the same way.
- Missing problems hidden in the tail, random samples, or rare categories.
- Treating `object` dtype as harmless when it may hide mixed strings, numbers, and null markers.
- Confusing row count with unique entity count.

Documentation references:
- [pandas essential basic functionality](https://pandas.pydata.org/docs/user_guide/basics.html)
- [pandas.DataFrame.info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html)
- [pandas.DataFrame.describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)


In [ ]:
display(sales_january_raw.head())
display(sales_february_raw.head())

inspection_summary = pd.DataFrame(
    [
        {
            "dataset": "sales_january_raw",
            "rows": len(sales_january_raw),
            "missing_values": int(sales_january_raw.isna().sum().sum()),
            "duplicate_rows": int(sales_january_raw.duplicated().sum()),
        },
        {
            "dataset": "sales_february_raw",
            "rows": len(sales_february_raw),
            "missing_values": int(sales_february_raw.isna().sum().sum()),
            "duplicate_rows": int(sales_february_raw.duplicated().sum()),
        },
        {
            "dataset": "products_df",
            "rows": len(products_df),
            "missing_values": int(products_df.isna().sum().sum()),
            "duplicate_rows": int(products_df.duplicated().sum()),
        },
    ]
)

january_missing = sales_january_raw.isna().sum().sort_values(ascending=False).rename("missing_count")
january_dtypes = sales_january_raw.dtypes.astype(str).rename("dtype")

display(inspection_summary)
display(january_dtypes.to_frame())
january_missing.to_frame()


## 3. Clean

Goal: make the data analysis-ready while preserving meaning and traceability.

Cleaning is not about making the table look pretty. It is about making rules explicit. A clean table has clear column names, reliable data types, consistent missing-value handling, standardized categories, and transparent business logic. The best cleaning steps are reproducible, easy to review, and tied to an observed issue from inspection.

Common cleaning operations:
- Standardize column names with `str.strip()`, `str.lower()`, and `str.replace()`.
- Rename columns to business-friendly names with `.rename()`.
- Convert types with `.astype()`, `.convert_dtypes()`, `pd.to_numeric()`, and `pd.to_datetime()`.
- Handle missing values with `.isna()`, `.fillna()`, `.dropna()`, `.where()`, or `.combine_first()`.
- Clean text with `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.replace()`, `.str.extract()`, and `.str.contains()`.
- Remove or diagnose duplicates with `.duplicated()` and `.drop_duplicates()`.
- Create derived columns with `.assign()` or direct column expressions.
- Standardize categories, units, date formats, and boolean flags.

Cleaning decisions worth discussing:
- Should invalid values raise errors, be coerced to missing, or be kept for manual review?
- Is it better to drop rows, fill values, or create a separate quality flag?
- Are some columns raw source fields that should be preserved untouched alongside cleaned versions?
- Should missing values be filled globally, per group, or not at all?
- Which rules are business rules and which are pure technical normalization?

Practical habits:
- Start from `clean_df = df.copy()` so the raw object stays available for comparison.
- Prefer vectorized transformations over manual loops.
- Keep a short note for each major cleaning rule so you can justify it later.
- Re-run inspection after major cleaning changes.

Common pitfalls:
- Hiding bad data by filling everything with one default value.
- Parsing dates without checking locale or day-month order.
- Mixing cleaned and raw categories in the same column.
- Dropping duplicates without first deciding what counts as a duplicate.

Documentation references:
- [pandas working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas.DataFrame.drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)
- [pandas.to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas.to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)


In [ ]:
def clean_sales_frame(frame: pd.DataFrame, source_month: str) -> pd.DataFrame:
    cleaned = frame.copy()
    cleaned.columns = (
        cleaned.columns.str.strip()
        .str.lower()
        .str.replace("%", "pct", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    text_columns = ["store_code", "product_code", "customer_segment", "sales_channel", "promo_flag"]
    for column in text_columns:
        cleaned[column] = cleaned[column].astype("string").str.strip()

    cleaned["store_code"] = cleaned["store_code"].str.upper()
    cleaned["product_code"] = cleaned["product_code"].str.upper()
    cleaned["customer_segment"] = cleaned["customer_segment"].replace({"": pd.NA}).str.title().fillna("Unknown")
    cleaned["sales_channel"] = cleaned["sales_channel"].str.title().fillna("Unknown")
    cleaned["promo_flag"] = cleaned["promo_flag"].str.lower().map({"yes": True, "no": False})

    parsed_dates = pd.to_datetime(cleaned["order_date"], errors="coerce", dayfirst=True)
    cleaned["order_date_invalid_flag"] = parsed_dates.isna()
    cleaned["order_date"] = parsed_dates

    units_numeric = pd.to_numeric(cleaned["units"], errors="coerce")
    cleaned["units_missing_flag"] = units_numeric.isna()
    cleaned["units"] = units_numeric.fillna(1).astype("Int64")

    cleaned["unit_price"] = pd.to_numeric(cleaned["unit_price"], errors="coerce")

    discounts = pd.to_numeric(cleaned["discount_pct"], errors="coerce")
    cleaned["discount_missing_flag"] = discounts.isna()
    cleaned["discount_pct"] = discounts.fillna(0)

    cleaned["source_month"] = source_month
    cleaned["gross_revenue"] = cleaned["units"].astype("float64") * cleaned["unit_price"]
    cleaned["net_revenue"] = (cleaned["gross_revenue"] * (1 - cleaned["discount_pct"])).round(2)
    cleaned["order_month"] = cleaned["order_date"].dt.strftime("%Y-%m")

    cleaned = cleaned.drop_duplicates(
        subset=["order_id", "order_date", "store_code", "product_code", "sales_channel"]
    ).reset_index(drop=True)
    return cleaned


sales_january_clean = clean_sales_frame(sales_january_raw, "2026-01")
sales_february_clean = clean_sales_frame(sales_february_raw, "2026-02")

cleaning_report = pd.DataFrame(
    [
        {
            "dataset": "sales_january_clean",
            "raw_rows": len(sales_january_raw),
            "clean_rows": len(sales_january_clean),
            "duplicates_removed": len(sales_january_raw) - len(sales_january_clean),
        },
        {
            "dataset": "sales_february_clean",
            "raw_rows": len(sales_february_raw),
            "clean_rows": len(sales_february_clean),
            "duplicates_removed": len(sales_february_raw) - len(sales_february_clean),
        },
    ]
)

display(cleaning_report)
sales_january_clean.head()


## 4. Filter

Goal: keep only the records and fields needed for the current question.

Filtering is the stage where you turn a large general-purpose table into a focused working set. It includes row filtering, column selection, sorting, ordering, and sometimes lightweight feature engineering that makes the subset easier to inspect or explain.

Common filtering patterns:
- Select columns directly with `df[[...]]`.
- Filter rows with boolean masks.
- Use `.loc[]` for label-based selection and `.iloc[]` for position-based selection.
- Combine conditions with `&`, `|`, and `~`.
- Use `.isin()` for membership tests.
- Use `.between()` for ranges.
- Use `.str.contains()` for text-based filters.
- Use `.query()` for SQL-like readability, especially in teaching.
- Use `.sort_values()`, `.sort_index()`, `.nlargest()`, and `.nsmallest()` to organize results.

Teaching prompts:
- Which rows are in scope for the analysis question?
- Which columns are inputs, outputs, labels, measures, or join keys?
- Is the filter a one-off exploration, or should it become part of the reusable pipeline?
- Would the logic be clearer as a boolean mask or a `query()` expression?

Important details:
- With boolean masks, use parentheses around each condition.
- With `query()`, use `@variable_name` to refer to Python variables.
- Sort before previewing if order matters for the story you want to tell.
- Prefer explicit column lists when handing a subset to later steps.

Common pitfalls:
- Chained filtering that becomes hard to read or debug.
- Forgetting parentheses around boolean conditions.
- Filtering before data types are fixed, especially for dates and numbers.
- Treating sorted output as if it changed the original table when it did not.

Documentation references:
- [pandas indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [pandas.DataFrame.query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html)


In [ ]:
analysis_columns = [
    "order_id",
    "order_date",
    "order_month",
    "store_code",
    "product_code",
    "customer_segment",
    "units",
    "unit_price",
    "discount_pct",
    "sales_channel",
    "promo_flag",
    "net_revenue",
    "source_month",
]


def filter_sales_frame(frame: pd.DataFrame) -> pd.DataFrame:
    filtered = frame.loc[
        frame["order_date"].notna()
        & frame["units"].gt(0)
        & frame["unit_price"].gt(0)
        & frame["sales_channel"].isin(["Online", "Retail"]),
        analysis_columns,
    ].copy()
    return filtered.sort_values(["order_date", "order_id"]).reset_index(drop=True)


sales_january_filtered = filter_sales_frame(sales_january_clean)
sales_february_filtered = filter_sales_frame(sales_february_clean)

allowed_channels = ["Online", "Retail"]
sql_style_preview = sales_february_clean.query(
    "sales_channel in @allowed_channels and unit_price > 0"
)[["order_id", "sales_channel", "unit_price", "net_revenue"]].head()

filter_report = pd.DataFrame(
    [
        {"dataset": "sales_january_filtered", "rows_after_filter": len(sales_january_filtered)},
        {"dataset": "sales_february_filtered", "rows_after_filter": len(sales_february_filtered)},
    ]
)

display(filter_report)
display(sql_style_preview)
sales_january_filtered.head()


## 5. Combine

Goal: integrate related data sources without losing control of row meaning.

Combining is where many pandas workflows become fragile. Row counts can explode, keys can mismatch, and null-heavy joins can silently hide data quality problems. This is why cleaning and inspection need to happen before serious merging.

Main combination patterns:
- `pd.concat()` for vertical stacking of similar extracts, such as monthly files.
- `pd.concat(..., axis=1)` for side-by-side alignment by index when that is intentional.
- `pd.merge()` for relational joins between fact and lookup tables.
- `.join()` as a convenience method for index-based joins.
- `pd.merge_asof()` for nearest-key or time-aware matching.
- `pd.merge_ordered()` for ordered data, often in time-oriented workflows.

Join choices worth demonstrating:
- `inner` join when you need only matched records.
- `left` join when one table is primary and you want to preserve all its rows.
- `right` join when the secondary table should define the final coverage.
- `outer` join when reconciliation is more important than row preservation simplicity.

Validation habits:
- Compare row counts before and after the merge.
- Check for duplicated keys on both sides before joining.
- Use `validate=` when you know the expected relationship, such as `1:1` or `m:1`.
- Use `indicator=True` when you want to diagnose unmatched records.
- Review null patterns in the newly joined columns.

Common pitfalls:
- Merging on keys with inconsistent types or formatting.
- Accidentally creating many-to-many joins and multiplying rows.
- Concatenating extracts with different schemas without checking column alignment.
- Assuming an outer join is safer when it may create a much harder table to interpret.

Documentation references:
- [pandas merge, join, concatenate and compare user guide](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas.merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
- [pandas.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)


In [ ]:
products_lookup = products_df.copy()
products_lookup["product_code"] = products_lookup["product_code"].astype("string").str.upper()

stores_lookup = stores_df.copy()
stores_lookup.columns = (
    stores_lookup.columns.astype("string")
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
stores_lookup["store_code"] = stores_lookup["store_code"].astype("string").str.upper()

sales_all = pd.concat([sales_january_filtered, sales_february_filtered], ignore_index=True)
sales_with_products = sales_all.merge(
    products_lookup,
    on="product_code",
    how="left",
    validate="m:1",
    indicator="product_match",
)
sales_enriched = sales_with_products.merge(
    stores_lookup,
    on="store_code",
    how="left",
    validate="m:1",
    indicator="store_match",
)
sales_analysis = sales_enriched.query(
    "product_match == 'both' and store_match == 'both'"
).copy()
sales_analysis["month"] = sales_analysis["order_date"].dt.strftime("%Y-%m")

merge_report = pd.DataFrame(
    [
        {"check": "rows_after_concat", "value": len(sales_all)},
        {"check": "product_match_both", "value": int((sales_enriched["product_match"] == "both").sum())},
        {"check": "product_match_left_only", "value": int((sales_enriched["product_match"] == "left_only").sum())},
        {"check": "store_match_both", "value": int((sales_enriched["store_match"] == "both").sum())},
        {"check": "store_match_left_only", "value": int((sales_enriched["store_match"] == "left_only").sum())},
        {"check": "rows_for_analysis", "value": len(sales_analysis)},
    ]
)

display(merge_report)
sales_analysis.head()


## 6. Summarize

Goal: turn many detailed rows into compact, decision-ready metrics.

Summarization is where raw records become insight. A good summary respects the table grain, groups by meaningful categories, uses measures that answer a question, and produces output that can be checked against business expectations.

Common summary patterns:
- `.groupby()` with `.sum()`, `.mean()`, `.median()`, `.min()`, `.max()`, `.count()`, `.size()`, or `.nunique()`.
- `.agg()` with multiple named aggregations so the output columns are readable.
- `.transform()` when you need group-level metrics back on each original row.
- `.value_counts()` for quick frequency summaries.
- `pd.crosstab()` for contingency-style tables.
- `pd.pivot_table()` for report-friendly, Excel-like summaries.
- Time-based summaries with `.resample()` when a datetime index or column is available.

Questions to ask before aggregating:
- What is the business unit of the measure: rows, customers, orders, revenue, hours, events?
- Do you need counts, sums, averages, distinct counts, or a mixture of them?
- Should missing categories be dropped or kept visible?
- Will the summary be used for reporting, charting, validation, or another merge?

Teaching opportunities:
- Compare `.count()` and `.size()` because they answer different questions.
- Show why named aggregations are easier to read than unnamed multi-level columns.
- Connect `pivot_table()` to what learners may already know from spreadsheet PivotTables.

Common pitfalls:
- Aggregating before you understand the row grain.
- Using averages where weighted logic or counts would be more honest.
- Producing summaries that cannot be traced back to the source rows.
- Forgetting to sort the final summary before presenting it.

Documentation references:
- [pandas group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [pandas.pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)
- [pandas.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)


In [ ]:
targets_clean = targets_df.copy()
targets_clean["month"] = targets_clean["month"].astype("string")
targets_clean["region"] = targets_clean["region"].astype("string")

category_summary = (
    sales_analysis.groupby(["month", "region", "category"], dropna=False)
    .agg(
        orders=("order_id", "nunique"),
        units_sold=("units", "sum"),
        revenue=("net_revenue", "sum"),
    )
    .reset_index()
    .sort_values(["month", "revenue"], ascending=[True, False])
)

region_month_summary = (
    sales_analysis.groupby(["month", "region"], dropna=False)
    .agg(
        orders=("order_id", "nunique"),
        units_sold=("units", "sum"),
        revenue=("net_revenue", "sum"),
        average_order_value=("net_revenue", "mean"),
    )
    .reset_index()
    .round(2)
)

region_target_summary = region_month_summary.merge(
    targets_clean,
    on=["month", "region"],
    how="left",
    validate="1:1",
)
region_target_summary["achievement_pct"] = (
    region_target_summary["revenue"] / region_target_summary["target_revenue"] * 100
).round(1)

revenue_pivot = pd.pivot_table(
    region_target_summary,
    index="region",
    columns="month",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
).round(2)

display(category_summary)
display(region_target_summary)
revenue_pivot


## 7. Reshape

Goal: convert the table into the layout needed for reporting, plotting, or downstream analysis.

Reshaping is the bridge between a technically correct table and a convenient table. Analysts often need long format for plotting and modeling, while managers often prefer wide format for reports. Knowing when to pivot and when to melt is one of the most useful workflow skills in pandas.

Core reshape tools:
- `pivot()` when each index-column pair has a single value.
- `pivot_table()` when duplicates need aggregation.
- `pd.melt()` when turning wide columns into row labels.
- `.stack()` and `.unstack()` for index-based reshaping.
- `.explode()` when a column contains list-like values that need row expansion.

When wide format helps:
- Side-by-side comparison by month, region, category, or scenario.
- Report tables that should look like spreadsheet summaries.
- Heatmap-style matrices built from summarized data.

When long format helps:
- Seaborn-style plotting where one column stores the measure and another stores the category.
- Multi-series visualizations where color, facet, or style comes from a variable column.
- Tidy workflows where each variable has one column and each observation has one row.

Common pitfalls:
- Using `pivot()` when duplicates exist and aggregation is actually required.
- Forgetting to rename `variable` and `value` columns after `melt()`.
- Reshaping too early and making cleaning harder.
- Building a wide table that becomes harder to merge or filter later.

Documentation references:
- [pandas reshaping and pivot tables user guide](https://pandas.pydata.org/docs/user_guide/reshaping.html)
- [pandas.melt](https://pandas.pydata.org/docs/reference/api/pandas.melt.html)
- [pandas.pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)


In [ ]:
report_wide = region_target_summary.pivot(index="region", columns="month", values="revenue").round(2)
metrics_long = region_target_summary.melt(
    id_vars=["region", "month"],
    value_vars=["revenue", "target_revenue", "achievement_pct"],
    var_name="metric",
    value_name="value",
).sort_values(["region", "month", "metric"]).reset_index(drop=True)

display(report_wide)
metrics_long.head(12)


## 8. Visualize

Goal: communicate an answer, not just produce a chart.

Visualization should come after the table is trustworthy enough to support a claim. In a pandas workflow, charts are usually based on cleaned and summarized data rather than directly on the raw source. This makes the visual easier to explain and much easier to validate.

Useful chart families to discuss:
- Line charts for trends over time.
- Bar charts for category comparison.
- Stacked or grouped bars for composition comparisons.
- Histograms for distribution shape.
- Box plots for spread and outliers.
- Scatter plots for relationships between two numeric fields.
- Heatmaps built from pivoted summaries.
- Small multiples or faceted views when one chart becomes too crowded.

Visualization tool choices:
- `DataFrame.plot()` and `Series.plot()` for quick pandas-native plots.
- `matplotlib` when you need fine control over axes, annotations, layouts, and styling.
- `seaborn` when a tidy long-format table is available and statistical defaults are helpful.

Chart selection cheat sheet:
- Use a bar chart when the question is "which category is larger?"
- Use a line chart when the question is "how did this change over time?"
- Use a histogram when the question is "how are values distributed?"
- Use a box plot when the question is "how do spread and outliers compare across groups?"
- Use a scatter plot when the question is "do two numeric variables move together?"
- Use a heatmap when the question is "where are the high and low pockets in a matrix?"

Preparation tips before plotting:
- Decide whether the chart should use raw rows, grouped summaries, or reshaped data.
- Sort the categories before plotting if order helps the message.
- Keep only the columns needed for the chart to reduce accidental confusion.
- Round or format values for labels after the calculation stage, not before it.
- Check whether missing values or filtered-out rows change the story.

Readability and design tips:
- Write titles that answer a business question rather than repeating the axis names.
- Use axis labels and units consistently.
- Keep legends short and place them where they do not block the data.
- Rotate labels only when necessary; if too many labels need rotation, reconsider the chart design.
- Use one highlight color intentionally and keep supporting colors quieter.
- Prefer direct value labels for short bar charts when exact numbers matter.
- Start bar charts from zero unless you have a very specific analytical reason not to.
- Use grids lightly; they should support reading, not dominate the figure.

Notebook workflow tips:
- Save important figures to files so the notebook produces reusable artifacts.
- Keep chart code close to the summary table it visualizes.
- In Colab, use explicit output paths because the runtime is temporary.
- In local mode, use repository output folders so exports stay versionable and easy to inspect.

Common pitfalls:
- Plotting too many categories in one chart.
- Using pie charts when bars or lines explain the comparison more clearly.
- Mixing incompatible scales without explanation.
- Spending too much time styling before the underlying summary is correct.
- Showing a chart without also checking the table behind it.
- Forgetting that the plotting stage can reveal mistakes in earlier cleaning or grouping logic.

Documentation references:
- [pandas chart visualization guide](https://pandas.pydata.org/docs/user_guide/visualization.html)
- [pandas.DataFrame.plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html)
- [Matplotlib plot types](https://matplotlib.org/stable/plot_types/index.html)
- [Matplotlib annotated heatmap example](https://matplotlib.org/stable/gallery/images_contours_and_fields/image_annotated_heatmap.html)
- [Matplotlib subplots guide](https://matplotlib.org/stable/gallery/subplots_axes_and_figures/subplots_demo.html)
- [Seaborn plotting function overview](https://seaborn.pydata.org/tutorial/function_overview.html)


In [ ]:
monthly_totals = region_target_summary.groupby("month")[["revenue", "target_revenue"]].sum().round(2)
category_totals = (
    sales_analysis.groupby("category")["net_revenue"].sum().sort_values(ascending=False).round(2)
)
channel_month = (
    sales_analysis.groupby(["month", "sales_channel"])["net_revenue"]
    .sum()
    .unstack(fill_value=0)
    .round(2)
)
achievement_heatmap = (
    region_target_summary.pivot(index="region", columns="month", values="achievement_pct")
    .fillna(0)
    .round(1)
)

category_colors = {
    "Electronics": "#4C78A8",
    "Office": "#F58518",
    "Breakroom": "#54A24B",
}

fig1, axes1 = plt.subplots(2, 2, figsize=(15, 10))

monthly_totals.plot(kind="bar", ax=axes1[0, 0], title="Actual Revenue vs Target by Month")
axes1[0, 0].set_xlabel("Month")
axes1[0, 0].set_ylabel("Revenue")
axes1[0, 0].tick_params(axis="x", rotation=0)
axes1[0, 0].grid(axis="y", alpha=0.25)

category_totals.plot(
    kind="bar",
    ax=axes1[0, 1],
    color=[category_colors.get(category, "#999999") for category in category_totals.index],
    title="Revenue by Product Category",
)
axes1[0, 1].set_xlabel("Category")
axes1[0, 1].set_ylabel("Revenue")
axes1[0, 1].tick_params(axis="x", rotation=20)
axes1[0, 1].grid(axis="y", alpha=0.25)
for patch in axes1[0, 1].patches:
    height = patch.get_height()
    axes1[0, 1].annotate(
        f"{height:.0f}",
        (patch.get_x() + patch.get_width() / 2, height),
        ha="center",
        va="bottom",
        xytext=(0, 4),
        textcoords="offset points",
        fontsize=9,
    )

channel_month.plot(
    kind="line",
    marker="o",
    linewidth=2,
    ax=axes1[1, 0],
    title="Monthly Revenue by Sales Channel",
)
axes1[1, 0].set_xlabel("Month")
axes1[1, 0].set_ylabel("Revenue")
axes1[1, 0].tick_params(axis="x", rotation=0)
axes1[1, 0].grid(axis="y", alpha=0.25)

sales_analysis.boxplot(column="net_revenue", by="sales_channel", ax=axes1[1, 1])
axes1[1, 1].set_title("Order Revenue Distribution by Channel")
axes1[1, 1].set_xlabel("Sales Channel")
axes1[1, 1].set_ylabel("Net Revenue")
axes1[1, 1].grid(axis="y", alpha=0.25)
fig1.suptitle("")

plt.tight_layout()
dashboard_path = OUTPUT_DIR / "day4_visualization_dashboard.png"
fig1.savefig(dashboard_path, dpi=150, bbox_inches="tight")
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(15, 5.5))

for category, subset in sales_analysis.groupby("category"):
    axes2[0].scatter(
        subset["units"],
        subset["net_revenue"],
        s=90,
        alpha=0.8,
        label=category,
        color=category_colors.get(category, "#999999"),
    )
axes2[0].set_title("Units vs Net Revenue")
axes2[0].set_xlabel("Units")
axes2[0].set_ylabel("Net Revenue")
axes2[0].grid(alpha=0.25)
axes2[0].legend(title="Category")

heatmap = axes2[1].imshow(achievement_heatmap.values, aspect="auto", cmap="YlGnBu")
axes2[1].set_title("Target Achievement Heatmap (%)")
axes2[1].set_xticks(range(len(achievement_heatmap.columns)))
axes2[1].set_xticklabels(achievement_heatmap.columns)
axes2[1].set_yticks(range(len(achievement_heatmap.index)))
axes2[1].set_yticklabels(achievement_heatmap.index)
for row_index in range(len(achievement_heatmap.index)):
    for col_index in range(len(achievement_heatmap.columns)):
        value = achievement_heatmap.iloc[row_index, col_index]
        axes2[1].text(
            col_index,
            row_index,
            f"{value:.0f}",
            ha="center",
            va="center",
            color="black",
            fontsize=9,
        )
fig2.colorbar(heatmap, ax=axes2[1], label="Achievement %")

plt.tight_layout()
examples_path = OUTPUT_DIR / "day4_visualization_examples.png"
fig2.savefig(examples_path, dpi=150, bbox_inches="tight")
plt.show()

visualization_outputs = pd.DataFrame(
    [
        {"artifact": "visual dashboard", "path": str(dashboard_path)},
        {"artifact": "extra examples", "path": str(examples_path)},
    ]
)
visualization_outputs


## 9. Export

Goal: deliver outputs that other people and later scripts can reuse.

Export is the point where a notebook stops being a private exploration and becomes part of a repeatable workflow. Good export choices depend on the audience. A colleague may want Excel, a pipeline may want parquet, a web service may want JSON, and a reporting layer may want a styled worksheet or HTML table.

Common export targets:
- `to_csv()` for universal exchange and simple downstream use.
- `to_excel()` for spreadsheet consumers and multi-sheet outputs.
- `to_parquet()` for efficient analytics storage and type preservation.
- `to_json()` for web-friendly interchange.
- `to_html()` for lightweight publishing.
- `to_sql()` for loading processed results back into a database.

Export decisions worth making explicit:
- Should the index be saved or reset first?
- Should the output contain cleaned detail rows, summary tables, or both?
- Are there multiple deliverables such as `cleaned`, `summary`, and `chart_ready` versions?
- Does the file name include a date, version, or workflow stage?
- Is compression useful for large text outputs?

Reproducibility habits:
- Export from named final DataFrames rather than from intermediate temporary objects.
- Keep output paths predictable and separate from raw input paths.
- Save summaries in a sorted and presentation-ready order.
- If the same export is reused often, turn it into a helper cell or function later.

Common pitfalls:
- Accidentally exporting the index as an unnamed extra column.
- Exporting wide reports that are hard to reuse programmatically when a long clean table should also be saved.
- Overwriting prior outputs without a naming convention.
- Delivering only a chart when the underlying summary table is also needed.

Documentation references:
- [pandas IO tools user guide](https://pandas.pydata.org/docs/user_guide/io.html)
- [pandas.DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)
- [pandas.DataFrame.to_excel](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html)
- [pandas.DataFrame.to_parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html)


In [ ]:
clean_output_path = OUTPUT_DIR / "sales_analysis_clean.csv"
summary_output_path = OUTPUT_DIR / "regional_target_summary.csv"
metrics_output_path = OUTPUT_DIR / "regional_metrics_long.json"
html_output_path = OUTPUT_DIR / "regional_revenue_report.html"

sales_analysis.to_csv(clean_output_path, index=False)
region_target_summary.to_csv(summary_output_path, index=False)
metrics_long.to_json(metrics_output_path, orient="records", indent=2)
report_wide.to_html(html_output_path)

if EXCEL_ENGINE:
    excel_output_path = OUTPUT_DIR / "day4_reporting_pack.xlsx"
    with pd.ExcelWriter(excel_output_path, engine=EXCEL_ENGINE) as writer:
        sales_analysis.to_excel(writer, index=False, sheet_name="sales_analysis")
        region_target_summary.to_excel(writer, index=False, sheet_name="region_targets")
        report_wide.reset_index().to_excel(writer, index=False, sheet_name="revenue_wide")
else:
    excel_output_path = None

artifact_rows = [
    {"artifact": "clean sales data", "path": str(clean_output_path)},
    {"artifact": "regional target summary", "path": str(summary_output_path)},
    {"artifact": "long metrics json", "path": str(metrics_output_path)},
    {"artifact": "html report", "path": str(html_output_path)},
    {"artifact": "excel pack", "path": str(excel_output_path) if excel_output_path else "skipped: no Excel engine installed"},
]

if "visualization_outputs" in globals():
    artifact_rows.extend(visualization_outputs.to_dict(orient="records"))

pd.DataFrame(artifact_rows)


## End-to-End Checklist

Before calling a pandas workflow complete, verify the following:
- The source loading choices are explicit and reproducible.
- The table grain is known and written down.
- Data types and missing values have been reviewed, not guessed.
- Cleaning rules are tied to observed issues.
- Filters reflect the analysis question and are readable.
- Merges and concatenations were validated with row counts and key checks.
- Summaries can be traced back to the source rows.
- Reshaping decisions support the reporting or visualization goal.
- Visuals answer a question clearly and are based on trustworthy tables.
- Exports are saved in formats that suit both humans and downstream code.


## Conclusion: Effective Workflow

An effective pandas workflow is not about memorizing isolated commands. It is about moving through a disciplined sequence where each stage prepares the next one: load carefully, inspect honestly, clean explicitly, filter intentionally, combine cautiously, summarize meaningfully, reshape for the task, visualize with purpose, and export for reuse.

When this workflow is done well, the notebook becomes more than a place to test code. It becomes a reproducible record of how raw information was turned into reliable analysis. That is the real value of pandas in practical work.
